# 08 — Baseline Forecasting

A strong baseline is a requirement: machine learning must beat simple strategies, not merely produce predictions.

In [ ]:

from pathlib import Path
import os, sys
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Project root:", ROOT)


In [ ]:
import pandas as pd, numpy as np
from src.utils.config import load_yaml
from src.forecasting.baselines import naive_forecast, seasonal_naive_forecast, moving_average_forecast
from src.forecasting.evaluate import evaluate
df=pd.read_parquet(ROOT/"data/processed/forecast_features.parquet"); cfg=load_yaml("model_config.yaml"); maxd=df.date.max(); test_start=maxd-pd.Timedelta(days=cfg["test_days"]-1); val_start=test_start-pd.Timedelta(days=cfg["validation_days"]); train=df[df.date<val_start]; val=df[(df.date>=val_start)&(df.date<test_start)]
rows=[]
for name,func in [("naive",naive_forecast),("seasonal_naive",seasonal_naive_forecast),("moving_average",moving_average_forecast)]:
    ys=[]; ps=[]
    for sid,g in val.groupby("id",observed=True):
        h=train[train.id==sid].sort_values("date")["demand"]; yt=g.sort_values("date")["demand"].to_numpy(); ys.extend(yt); ps.extend(func(h,len(yt)))
    rows.append({"model":name,**evaluate(ys,ps)})
print(pd.DataFrame(rows).sort_values("WAPE"))

In [ ]:
sid=sorted(val.id.astype(str).unique())[0]; h=train[train.id.astype(str)==sid].demand; actual=val[val.id.astype(str)==sid].sort_values("date").demand; p=seasonal_naive_forecast(h,len(actual));
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,4)); ax.plot(actual.to_numpy(),label="actual"); ax.plot(p,label="seasonal naive"); ax.legend(); ax.set_title(f"Baseline example: {sid}"); plt.show()

Interpretation should focus on whether weekly repetition helps, whether intermittent zeros are handled, and which baseline is difficult for ML to beat.